# Lab 11: Debugging Agents from Production Traces

Your team runs a coding agent that takes GitHub issues and submits patches:
[mini-swe-agent](https://github.com/SWE-agent/mini-swe-agent), an LLM in a loop that can only run bash commands,
driven by **Gemini 3.5 Flash**. These are **real, public runs** on
[SWE-bench Verified](https://www.swebench.com/) (real issues from real Python repos), published by the SWE-bench
team. Every run is recorded as a trace: each model call, each shell command, its output and its token count. Each
patch was also graded by the maintainers' hidden tests, a ground truth you rarely get in production.

Leadership wants to know whether to give the agent more autonomy and budget. You'll answer five questions using
**[Arize Phoenix](https://github.com/Arize-ai/phoenix)**, an open-source tool for tracing and evaluating LLM
applications, plus a little pandas:

| Part | Question | Time |
|---|---|---|
| 1 | What does a run look like, and what does default tracing miss? | 8 min |
| 2 | How good is this agent, and is it the right model? | 5 min |
| 3 | Why does it fail? | 15 min |
| 4 | What does it cost, and where does the money go? | 7 min |
| 5 | Can we trust an LLM judge to grade it? | 15 min |

Phoenix holds **40 of the agent's runs**, hand-picked to include more failures than usual. The full submission
solved 71.8% of SWE-bench Verified's 500 tasks; it has runs for 441 of them (the other 59 count as unresolved), and
those 441 runs resolve 81%. Parts 2 and 4 also use summary data for all runs.

**Before you start:** in a terminal, run `phoenix serve` and keep it running. Open http://localhost:6006.
Cells marked **TODO** are yours; questions marked **Q** are for your checkoff.

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd

import sys

import helpers as h

pd.set_option("display.max_colwidth", 100)
!{sys.executable} load_traces.py

## Part 1: Read a trace (8 min)

Every model call the agent makes goes through the LiteLLM library, so the quickest way to trace it is an
off-the-shelf **auto-instrumentation** for LiteLLM.

**1a.** In Phoenix, open the project **lab11-part1-autotrace**: one agent run, recorded with auto-instrumentation
only.

**1b.** Open **lab11-swebench** and paste this into the filter bar to see the same run with full instrumentation
(the agent loop, each step and each shell command have their own spans):

In [ ]:
print(f"agent.trace_id == '{h.part1_run()}'")
# This run is long (121 steps). To jump to the commands Q2 asks about, use this filter instead:
print(f"span_kind == 'TOOL' and agent.trace_id == '{h.part1_run()}' and agent.step_index in (82, 83, 94, 103)")

**Q1.** How many traces did one run produce in the auto-instrumented project? Open a few. Auto-instrumentation
does record each call's messages, so you can find commands inside them. But could you filter for the commands that
failed, see which step of the run a call belongs to, or tell whether the run fixed the issue?

**Q2.** Steps 82, 83 and 94 all run the same `queries` test suite. Which are marked as errors (red), which as OK
(green), and why? Read the commands carefully, and look at what step 83's output actually says. At step 103 the
agent sees "ALL TESTS PASSED" and submits. By then it really had fixed the tests it broke, but the hidden tests
still failed: look at the root span's `eval.*` attributes. What did its own checks not cover? If your team's
dashboard showed "% of tool calls that errored", what would it have missed in this run?

**Q3.** The instrumentation is in `optional/tracing.py`. Read `agent_span()`. mini-swe-agent ends a run by raising
a `Submitted` exception, and OpenTelemetry marks any span an exception passes through as an error. What does the
code do with that exception, and what does it do with any other exception? What would an alert on "steps that
raised an exception" show if it didn't make that distinction? Separately: 36 of the 40 runs contain at least one
red command span, simply because some command exited non-zero. Why is "the run contains an error" a poor health
metric for an agent?

## Part 2: How good is it, and is it the right model? (5 min)

The SWE-bench team runs many models with the same agent framework (mini-swe-agent v2.0; our Gemini 3.5 Flash runs
used v2.4). `h.model_results()` has every model's result and cost on all 500 tasks (one attempt each).

**TODO 1:** for each model, compute the resolve rate, the average cost per task, and the cost per *resolved*
task. Gemini 3.5 Flash has no recorded cost in the public data: make sure it shows up as missing (NaN), not as $0
(hint: `sum(min_count=1)`).

In [ ]:
results = h.model_results()

# TODO 1: one row per model with columns resolve_rate, cost_per_task, cost_per_resolved
by_model = None
by_model

In [ ]:
# What if you could run several models on each task and keep whichever one passes?
solved = results.pivot(index="instance_id", columns="model", values="resolved")
cheap = ["GPT-5 mini", "MiniMax M2.5", "Kimi K2.5"]
print(f"solved by at least one model:     {solved.any(axis=1).mean():.0%}")
print(f"solved by every model:             {solved.all(axis=1).mean():.0%}")
print(f"solved by at least one cheap model: {solved[cheap].any(axis=1).mean():.0%}  "
      f"at ${results[results.model.isin(cheap)].cost_usd.sum() / 500:.2f} per task")

These ideas have names: **pass@k** is the chance that at least one of k attempts succeeds; **pass^k** is the
chance that all k succeed. Here the k attempts are different models; with one model you'd run it k times.
pass^k is the reliability users see when they rely on the agent again and again with nothing to catch a bad
attempt.

**Q4.** Which single model would you ship, and why? Running three cheap models and keeping a passing answer beats
the best single model at about a third of its cost. What do you need for that to work in production, where there
are no hidden tests to tell you which answer passed? (Hold that thought for Part 5.) When would pass^k be the number
that matters instead?

## Part 3: Why does it fail? (15 min)

You can't read every failed trace in full. Debugging at scale is three moves: **search** across runs for patterns,
**deep-dive** one run to find where it first went wrong, and **generalize** what you looked for into labels on every
run: a failure taxonomy that tells you what to fix first.

### 3a. Search
Try these in the **lab11-swebench** filter bar:

| Filter | Shows you |
|---|---|
| `span_kind == 'TOOL' and status_code == 'ERROR'` | every failed shell command across all runs |
| `eval.resolved == False` | runs whose patch failed the hidden tests |
| `eval.pass_to_pass_failed > 0` | runs that broke tests that used to pass |
| `agent.n_steps > 80` | the longest runs |
| `span_kind == 'TOOL' and 'git checkout' in input.value` | the agent undoing its own edits |

### 3b. Deep-dive
Find your assigned failed run:

In [ ]:
ANDREW_ID = "your_andrew_id"   # TODO: put your Andrew ID here

runs = h.runs()                       # one row per run in Phoenix: agent.*, eval.* attributes of the root span
failed = runs[~runs["eval.resolved"]].sort_values("agent.trace_id")
mine = failed.iloc[sum(map(ord, ANDREW_ID)) % len(failed)]
print(f"agent.trace_id == '{mine['agent.trace_id']}'")

**Q5.** Open your run. Read the issue (the root span's input), then the agent's last few steps and its patch.
Where did it first go wrong (often long before the end)? Did it misunderstand the issue, fix only the example in
the issue, break something else, or run out of ideas? Is that a **model** problem, a **harness** problem (tools,
prompt, budget) or a **task** problem (an ambiguous issue)?

### 3c. Generalize
Turn what you looked for into detectors. Each gets the list of shell commands one run executed, in order.
`h.is_edit_command(cmd)` and `h.is_test_command(cmd)` are provided.

**TODO 2:** implement the two detectors.

In [ ]:
def stuck_in_loop(cmds, k=3):
    """True if the run executed the exact same command (ignoring whitespace) k or more times in total
    (not necessarily in a row)."""
    # TODO 2a
    raise NotImplementedError


def verified_after_last_edit(cmds):
    """True if the run ran a test or reproduction command AFTER its last source edit.
    False if the run never edited anything."""
    # TODO 2b
    raise NotImplementedError

In [ ]:
# A detector is only useful if it separates failures from successes. How often does each one fire?
cmds_by_run = h.tool_calls().groupby("otel_trace_id")["input"].apply(list)
cmds = runs["otel_trace_id"].map(lambda t: cmds_by_run.get(t, []))
check = pd.DataFrame({"resolved": runs["eval.resolved"],
                      "stuck_in_loop": cmds.map(stuck_in_loop),
                      "verified_after_last_edit": cmds.map(verified_after_last_edit),
                      "80+ steps": runs["agent.n_steps"] > 80})
check.groupby("resolved").mean().T.rename(columns={False: "fires on failed runs", True: "fires on resolved runs"})

In [ ]:
def classify_failure(run, cmds):
    """One label per failed run. Order matters: the first rule that matches wins."""
    if (run.get("eval.pass_to_pass_failed") or 0) > 0:
        return "broke existing tests"
    if run["agent.n_steps"] > 80:
        return "long struggle (80+ steps)"
    if not verified_after_last_edit(cmds):
        return "wrong fix, never verified"
    return "wrong fix, self-check passed"


failure_mode = failed.apply(lambda r: classify_failure(r, cmds_by_run.get(r["otel_trace_id"], [])), axis=1)
h.annotate(failure_mode, name="failure_mode")      # push the labels back into Phoenix
failure_mode.value_counts().plot.barh(title=f"Failure modes ({len(failed)} failed runs)").invert_yaxis()
plt.tight_layout(); plt.show()

Reload Phoenix: each failed run now carries a `failure_mode` annotation. Try
`annotations['failure_mode'].label == 'wrong fix, self-check passed'`.

**Q6.** Which rows of this table actually separate failed runs from resolved ones? Why do plausible heuristics like
"stuck in a loop" fire just as often on successes? Now be suspicious of the one that looks perfect: these 40 runs
were hand-picked. Check the `80+ steps` row on all 441 runs (`h.all_runs()`; the number of steps is the length
of `input_tokens_per_step`). What changes, and what does that tell you about validating heuristics on a small,
selected sample?

**Q7** *(if time allows)*. The biggest category is "wrong fix, self-check passed": the agent ran tests, they
passed, and the hidden tests still failed. Open one. Compare what the agent tested with what the issue asked for. Is this fixable in the
harness, or does it need a better model?

## Part 4: What does it cost? (7 min)

Every model call in the trace carries its token counts (`llm.token_count.*`) and an estimated cost
(`llm.cost.total`). Click a few LLM spans in your Part 1 run and watch the prompt token count as the run goes on.

The public runs recorded tokens but not dollars, so costs here are tokens times Google's list price for Gemini
3.5 Flash (see `data/lab_config.json`). For cost questions, use **all** 441 runs, not just the 40 in Phoenix.
(The 59 tasks with no run have no token data, so they're left out of the cost numbers.)

In [ ]:
cfg = h.lab_config()
price_in, price_out = cfg["price_in_per_million"] / 1e6, cfg["price_out_per_million"] / 1e6
everything = h.all_runs()     # one row per run, with input/output tokens for every step
everything["n_steps"] = everything.input_tokens_per_step.map(len)
everything["cost"] = everything.input_tokens_per_step.map(sum) * price_in + everything.output_tokens_per_step.map(sum) * price_out
print(f"{len(everything)} runs, {everything.resolved.mean():.0%} resolved, ${everything.cost.sum():.2f} total")

# The agent resends its whole history on every call, so later calls cost more:
step_cost = pd.DataFrame([(i // 10 * 10, x * price_in + y * price_out)
                          for ins, outs in zip(everything.input_tokens_per_step, everything.output_tokens_per_step)
                          for i, (x, y) in enumerate(zip(ins, outs))], columns=["step", "cost"])
step_cost.groupby("step").cost.mean().plot.bar(title="Average cost of one model call vs. step number")
plt.xlabel("step (bucketed by 10)"); plt.ylabel("$ per call"); plt.tight_layout(); plt.show()

**TODO 3:** compute the cost per resolved task, and where the money goes: what share of runs vs. what share of
spend is resolved runs, failed runs, and long runs (more than 80 steps)? The groups overlap (a long run is also
resolved or failed), so the shares won't add up to 1. How does our agent's cost per resolved task compare with
the models in Part 2?

In [ ]:
# TODO 3
cost_per_resolved = None
spend = None    # DataFrame: one row per group (resolved / failed / 80+ steps), columns share_of_runs and share_of_spend
print(f"cost per resolved task: {cost_per_resolved}")
spend

**TODO 4:** the agent never sees its step limit, so you can simulate a lower limit `L`: a run resolves under `L`
only if it resolved within `L` steps, and it costs only its first `L` model calls.

In [ ]:
rows = []
for L in [30, 40, 50, 60, 70, 80, 100, 130]:
    # TODO 4
    resolved_within_L = None   # number of runs that resolved using at most L steps
    cost_within_L = None       # total $ if every run were stopped after L model calls
    rows.append({"step_limit": L, "resolved": resolved_within_L, "cost": cost_within_L})
budget = pd.DataFrame(rows)
budget

**Q8.** Long runs are a small share of runs but a large share of spend, and they resolve less often. What step
limit would you ship, how many resolves would you give up, and how much would you save? There's no free lunch
here, so justify the trade. Name one reason the simulation could be wrong in production.

## Part 5: Can we trust the judge? (15 min)

In production there are no hidden tests. Teams often add an **LLM judge**: a second model reads the trace and the
patch and says "resolved" or "not resolved". It's also exactly the verifier Part 2's "run three models, keep the
one that passes" idea needs. Here we still have the hidden-test results, so we can grade the grader, and you'll
build and improve the judge inside Phoenix rather than in code.

`load_traces.py` already set up three things in Phoenix:

| In Phoenix | What it is |
|---|---|
| **Datasets → `lab11-judge-set`** | 40 examples, one per run. Input: the issue, a condensed transcript, the patch. Reference output: the hidden tests' verdict. Each example links back to its run's trace. |
| **Prompts → `lab11-judge`** | Version 1 of the judge: a reasonable first try. |
| **`verdict_correct`** (an evaluator on the dataset) | Checks the judge's final `VERDICT:` line against the hidden tests: 1 if right, 0 if wrong. |

**Setup:** the judge calls Gemini with your course credits. Add your key in Phoenix under **Settings → AI
Providers → Google**, or stop Phoenix, run `export GEMINI_API_KEY=...`, and start `phoenix serve` again in the same
terminal (your traces are kept). Each run of the judge over the dataset costs a few cents.

### 5a. Run the starter judge
1. Open **Prompts → `lab11-judge`** and click **Playground**. Check the model is Google / `gemini-3.5-flash-lite` (a
   different model from the agent's, so the judge isn't grading its own work).
2. Select the dataset **`lab11-judge-set`**. The template variables `{{issue}}`, `{{transcript}}` and `{{patch}}`
   now fill from each example.
3. Turn on the **`verdict_correct`** evaluator, name the experiment **`judge-v1`**, and click **Run**. Phoenix calls
   the judge once per example (about a minute) and records the results as an experiment on the dataset.
4. Open **Datasets → `lab11-judge-set` → Experiments**. The `verdict_correct` score is the judge's accuracy.

Accuracy hides which way the judge is wrong, and the two directions cost very different amounts.

**TODO 5:** write `score()`, which turns one judge experiment into precision and recall for "resolved".

In [ ]:
def score(experiment):
    r = h.judge_results(experiment)    # no API key? use h.judge_results("precomputed") if your TA provided it
    says = r["judge_says"] == "resolved"
    truth = r["hidden_tests"] == "resolved"
    # TODO 5
    precision = None   # when the judge says resolved, how often is it right?
    recall = None      # of the truly resolved runs, how many does the judge catch?
    return pd.Series({"precision": precision, "recall": recall, "says resolved": says.sum(),
                      "no verdict": (r["judge_says"] == "no verdict").sum()})


v1 = h.judge_results("judge-v1")
print(pd.crosstab(v1["judge_says"], v1["hidden_tests"], rownames=["judge says"], colnames=["hidden tests"]))
score("judge-v1")

### 5b. Read what fooled it
In the `judge-v1` experiment, find the rows where `verdict_correct` is 0 and the reference verdict is
`not_resolved`: these are **false positives**, broken patches the judge passed. Open one. Read the judge's
reasoning, then follow the example's link back to the run's trace and look at what the agent actually tested.
Note what the judge sees: the `transcript` input holds only the agent's last 25 steps, with long outputs cut.

**Q9.** What fooled the judge, and how is that related to Q7? (Your `failure_mode` annotations from Part 3 are on
the same runs.)

### 5c. Improve the judge
Edit the prompt in the Playground to fix what you saw. Ideas: tell the judge the agent's own test runs may test the
wrong thing; ask it to check whether the tests the agent ran are the repo's existing tests for the code it
changed; ask what a skeptical reviewer would check before merging. Save it as a new version of `lab11-judge`, run
it over the dataset as **`judge-v2`**, then select both experiments and **Compare** them to see which examples
flipped, in which direction.

In [ ]:
# Compare every judge experiment you've run (add more names as you iterate)
pd.DataFrame({name: score(name) for name in ["judge-v1", "judge-v2"]}).T.round(2)

**Q10.** Did your change buy precision, and what did it cost in recall? You tuned `judge-v2` while looking at
these same 40 runs. Why is its score optimistic, and how would you get an honest number? Would you let this judge
**auto-merge** patches, or only **triage** them for human review? How would you keep measuring it once there are
no hidden tests? And is judging every run affordable? Each experiment row shows its prompt tokens; the judge's
price is in `h.lab_config()`. Compare with what one agent run costs (Part 4).

## Wrap-up questions
**Q11.** These traces contain full prompts, source code and command outputs. In a company, what could leak through
them, and what would you redact, sample, or delete after a while?

**Q12.** At a million agent runs a day you can't keep every trace. Which ones would you keep, and why?

## Checkoff
Show a TA:
1. The auto-instrumented vs. full trace of the Part 1 run, and your answers to **Q1** to **Q3**.
2. Your model comparison and your answer to **Q4**.
3. Your assigned run's first wrong step and root cause (**Q5**), the detector check (**Q6**), your failure-mode
   chart, a Phoenix filter on your `failure_mode` annotation, and **Q7**.
4. Where the money goes and your step limit with its trade-off (**Q8**).
5. Your `judge-v1` and `judge-v2` experiments in Phoenix, their precision and recall, one false positive
   explained (**Q9**), and your deploy decision (**Q10**).
6. Short answers to **Q11** and **Q12**.